# Combined Pipeline Notebook

This notebook combines all Python modules from the FP project in dependency order. Each section corresponds to a separate Python file from the codebase.

**Execution Order:**
1. log.py - Logging utilities
2. prompts.py - Prompt templates
3. io_utils.py - File I/O utilities
4. patch_output.py - Data models for patches
5. vector_store.py - Vector store management
6. github_utils.py - GitHub API utilities
7. process_layer.py - LLM processing layer with interactive model selection
8. apply_patch.py - Patch application utilities
9. offline_pipeline.py - Pipeline orchestration
10. bugsinpy_bugs.py - BugsInPy processing
11. swe_bench.py - SWE-bench evaluation
12. **Run SWE-bench** - Execute the main function

In [ ]:
# from google.colab import driv
# drive.mount('/content/drive')

In [ ]:
!sudo apt update
!sudo apt install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

In [4]:
EMBEDDING = "granite-embedding:latest"
NUM_THREAD = 6
CONTEXT = 512

DRIVE_PATH="drive/MyDrive/final-project-code/code"
LLM_NAME="gpt-5-nano"

BATCH_SIZE=512
CHUNK_SIZE=400

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
# !ollama pull qwen3-embedding:0.6b
# !ollama pull unclemusclez/jina-embeddings-v2-base-code
# !ollama pull vuongnguyen2212/CodeRankEmbed
# !ollama pull embeddinggemma:300m
# !ollama pull nomic-embed-text
# !ollama pull manutic/nomic-embed-code
# !ollama pull starcoder2
# !ollama pull dolphincoder
!ollama pull granite-embedding:latest
# !ollama pull gemma3:12b
# !ollama pull codegemma:7b-instruct
# !ollama pull codellama:instruct
# !ollama pull llama3.2:3b
# !ollama pull deepseek-r1:8b
# !ollama pull granite3.1-dense:8b
# !ollama pull granite3.3:8b
# !ollama pull phi4-mini:3.8b
# !ollama pull cogito:8b
# !ollama pull llama3.1:8b
# !ollama pull mistral:7b-instruct
# !ollama pull gemma3n:e4b

# !ollama pull cogito:14b
# !ollama pull gpt-oss:20b
# !ollama pull magistral:24b
# !ollama pull cogito:32b
# !ollama pull gemma3:27b 

# Large ones
# !ollama pull codebooga:34b
# !ollama pull llama4:16x17b
# !ollama pull mistral-large:123b
!ollama pull gpt-oss:120b
# !ollama pull cogito:70b
# !ollama pull llama3.1:70b

In [ ]:
%pip install --upgrade pip
%pip install -r requirements.txt
# Downgrade protobuf to a compatible version
%pip install protobuf==3.20.3

Note: you may need to restart the kernel to use updated packages.
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiohttp-3.13.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached anyio-4.12.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-manylinux_2_34_x86_64.whl.metadata (10 kB)
  Using cached build-1.4.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached cachetools-6.2.3-py3-none-any.whl.metadata (5.6 kB)
  Using cached certifi-2025.11.12-py3-none-any.whl.metadata (2.5 kB)
  Using cached cffi-2.0.0-cp313-cp313-manylinux2014_x86_64.ma

In [ ]:
from __future__ import annotations

import getpass
import hashlib
import json
import os
import re
import subprocess
import threading
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timedelta
from difflib import unified_diff
from pathlib import Path
from time import sleep
from typing import Any, Dict, Iterable, List, Literal, Optional, Set, Tuple
from urllib.parse import urlparse
from hashlib import sha256

import chromadb
import difflib
import jsonlines
import requests
from datasets import load_dataset
from github import Auth, Github, GithubRetry
from github.Issue import Issue
from langchain_chroma import Chroma
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.messages import AIMessage
from langchain_core.vectorstores.base import VectorStore
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

/home/arshia2562/Documents/Uni/Term-8/FP/.notebook/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.environ["GOOGLE_API_KEY"]=""
os.environ["GITHUB_TOKEN"]=""
os.environ["HUGGINGFACEHUB_API_TOKEN"]=''
os.environ["OPENROUTER_API_KEY"]=""
os.environ["JINA_API_KEY"]=""
os.environ["AVALAI_KEY"]=""

In [ ]:
# LLM = ChatOllama(model=LLM_NAME, temperature=0, top_p=0.8)

LLM = ChatOpenAI(
    api_key=os.environ.get("AVALAI_KEY"),
    base_url="https://api.avalai.ir/user/v1/chat/completions",
    model=LLM_NAME,
)

embeddings = OllamaEmbeddings(
    model=EMBEDDING, num_thread=NUM_THREAD, 
    num_ctx=CONTEXT, 
    base_url="127.0.0.1:11434", 
    num_gpu=-1, validate_model_on_init=True
)
# embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
# embeddings = FakeEmbeddings(size=1352)
# embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5")


## 1. log.py - Logging Utilities

In [6]:
filename = f"log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

def log_and_print(msg):
    os.makedirs("logs", exist_ok=True)
    with open("logs/" + filename, "a+") as f:
        f.write(str(msg) + "\n")
    print(msg)

## 2. prompt.py

In [ ]:

USER_PROMPT_MULTIFILE_ZERO_SHOT = """
You will be provided with an issue statement explaining a problem to resolve, and related context from repository that can be files from code base or other simillar issues.
I need you to solve this issue by generating a single patch file that I can apply directly to this repository using git apply.
"""

USER_PROMPT_MULTIFILE_COT = USER_PROMPT_MULTIFILE_ZERO_SHOT + """
step 1: read the given issue statement and its related context
step 2: understand the issue statment and what it's trying to achieve
step 3: find the root cause of the issue in the given context from the repository
step 4: produce a list of edit suggestions that completely fixes the issue with the smallest safe changes.
"""


OUTPUT_FORMAT_Diffview = """
OUTPUT FORMAT:
Output ONLY a single JSON object that conforms to this schema:
  DiffViewEdits = {
    "patch": string
  }
- No extra keys. No surrounding text. No markdown. No code fences. No explanations.
- All strings must be valid JSON strings (escape newlines as \\n and quotes as \\").
- The "patch" value MUST be a valid unified diff for `git apply`.

CRITICAL RULES for generating the unified diff:
1. Start each file with `diff --git a/<path> b/<path>`.
2. Include `--- a/<path>` and `+++ b/<path>` lines.
3. Use hunk headers: `@@ -<old_start>,<old_count> +<new_start>,<new_count> @@`
   - old_count = total lines from old file in this hunk (context + removed lines)
   - new_count = total lines in new version of this hunk (context + added lines)
4. Context lines (unchanged) MUST start with a single space ' ' and be copied
   EXACTLY character-for-character from the provided file content, preserving
   all indentation, whitespace, and punctuation. Never alter context lines.
5. Removed lines start with '-' followed by the EXACT original line content.
6. Added lines start with '+' followed by the new content.
7. Include 3 lines of unchanged context before and after each change.
8. Every hunk MUST be complete — never truncate or leave partial lines.
9. Use correct line numbers matching the provided file content.
10. Do NOT include any trailing text, markdown fences, or explanations after the diff.

Example (single hunk, 3 context lines before and after):
diff --git a/src/utils.py b/src/utils.py
--- a/src/utils.py
+++ b/src/utils.py
@@ -10,7 +10,7 @@
     results = []
     for item in data:
         if item is not None:
-            results.append(str(item))
+            results.append(item.strip())
         else:
             results.append("")
     return results
"""


OUTPUT_FORMAT_FILESTOEDIT = """
OUTPUT FORMAT:
Output ONLY a single JSON object that conforms to this schema:
  FilesToEdit = {
    "files_for_editing": [
      "file_path_1",
      "file_path_2",
      "file_path_3"
    ],
  }
- No extra keys. No surrounding text. No markdown. No code fences.
- All strings must be valid JSON strings (escape newlines as \n and quotes as \").
- return ONLY 3 items in files_for_editing
"""

USER_PROMPT_FILESTOEDIT = """
Given the issue statement and its top 30 similar files in code base as file skeletons,
your task is to select ONLY 3 files you think that are most related to the issue and need to be edited in order to solve the issue.
you should ONLY select from the top 30 files given to you.
The file skeletons includes the module
docstring (if available). It also contains class names,
their associated docstrings, and all method names. For
functions, only the name and the first/last five lines of
code are included.
"""

SYSTEM_PROMPT_PYTHON_PROGRAMMER = """You are an expert python programmer. Your task is to review and solve github issues based on repository context."""

SYSTEM_PROMPT_EMPTY = ""


DISCUSSION_SUMMARY_PROMPT = """You are analyzing a GitHub issue discussion to help with Automated Program Repair (APR).

Issue Title: {issue_title}
Issue Body: {issue_body}

Discussion Comments:
{comments}

Please provide a concise technical summary that includes:

1. **Bug Description**: What is the specific bug or problem reported?
2. **Symptoms**: How does the bug manifest? (error messages, unexpected behavior, etc.)
3. **Root Cause**: What is causing this issue? (if discussed)
4. **Affected Components**: Which files, functions, or code areas are affected?
5. **Proposed Solutions**: What fixes or workarounds were suggested or discussed?
6. **Resolution Status**: Was the issue resolved? How?

Focus on technical details relevant to code repair. Be specific about code locations, error messages, and implementation details.

Summary:"""


## 3. io_utils.py - File I/O Utilities

In [ ]:
TEXT_EXTS = {
    ".py"
}


def get_file_content(path: str) -> str:
    """
    Returns [str] file text content from [str] a path.
    Accepts both absolute paths and relative paths (which will be prefixed with 'codebase/').
    """
    p = Path(path)
    if not p.is_absolute() and not p.exists():
        p = Path("codebase") / path
    return p.read_text(encoding="utf-8")


def iter_text_files(codebase_root: str) -> Iterable[Path]:
    log_and_print(f"Checking project root dir: {codebase_root}")
    root = Path(codebase_root)
    for p in root.rglob("*"):
        log_and_print(f"Checking file: {p.as_posix()}")
        if p.is_dir():
            continue

        if any(part in {".git", "__pycache__", ".venv", "venv", "node_modules"} for part in p.parts):
            continue
        if p.suffix.lower() in TEXT_EXTS or p.name.lower() in {"dockerfile"}:
            log_and_print(f"Yielding file: {p.as_posix()}")
            yield p


def iter_issues_jsonl() -> Iterable[Any]:
    with jsonlines.open("issues/issues.jsonl") as reader:
        for issue in reader:
            yield issue


def get_issue_content(issue_number: int, issues_path_jsonl: str = "issues/issues.jsonl") -> tuple[str | None, str | None, str | None]:
    """
    Returns tuple[str | None, str | None, str | None] issue text content from [int] an id:
    (issue["title"], issue["body"], issue["labels"]) 
    """
    with jsonlines.open(issues_path_jsonl) as reader:
        for issue in reader:
            if issue["number"] == issue_number:
                return (issue["title"], issue["body"], issue["labels"])
                
    return (None, None, None)


def get_issues_comments_content(issue_number: int) -> list[tuple[str, str]]:
    """
    Returns list[tuple[str, str]] comments under and issue for [int] an issue number:
    (comment["commment_id"], comment["body"]) 
    """
    comments: list[tuple[str, str]] = []
    with jsonlines.open("commments.jsonl") as reader:
        for comment in reader:
            if comment["issue_number"] == issue_number:
                comments.append((comment["commment_id"], comment["body"]))
                
    return comments


def get_project_tree(project_root: str) -> str:
    try:
        tree = subprocess.run(
            ["tree", "--dirsfirst", "-anqf", "--noreport", "--gitignore", "-P", "*.py", "--prune"],
            cwd=project_root,
            check=True,
            capture_output=True,
            text=True,
        )

        print(f"✓ Successfully got project tree")
        return tree.stdout
    except subprocess.CalledProcessError as e:
        print(f"Could not get project tree: {e.stderr.strip() if e.stderr else str(e)}")
        return ""
    except Exception as e:
        print(f"Project tree recieve failed: {e}")
        return ""

## 4. patch_output.py - Data Models

**Note**: See remaining modules in cells below. Each module is fully self-contained.

In [ ]:
class DiffViewEdits(BaseModel):
    patch: str


class FilesToEdit(BaseModel):
    files_for_editing: List[str] = Field(default_factory=list)

## 5. vectore_store.py

In [ ]:
def make_id(text):
    return sha256(text.encode("utf-8")).hexdigest()


def build_vector_store(codebase_root: str="codebase", issues_jsonl_path: str | None="issues/issues.jsonl", batch_size: int = BATCH_SIZE, splitter_chunk_size: int = CHUNK_SIZE) -> VectorStore:
    collection_id = f"{codebase_root}|{issues_jsonl_path}"
    collection_name = f"collection_{make_id(collection_id)[:16]}"
    log_and_print(f"Using collection: {collection_name} (for {codebase_root}, {issues_jsonl_path})")
    
    # client = chromadb.HttpClient(host="127.0.0.1", port=8088, ssl=False)
    vector_store_from_client = Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        persist_directory="chroma_stuff"
    )
    # vector_store_from_client.reset_collection()
    existing_count = vector_store_from_client._collection.count()
    if existing_count > 0:
        log_and_print(f"Collection '{collection_name}' already has {existing_count} documents. Skipping reprocessing.")
        return vector_store_from_client
    
    log_and_print(f"Collection '{collection_name}' is empty. Processing documents...")
    
    docs: List[Document] = []
    for p in iter_text_files(codebase_root):
        log_and_print(f"Name of the file scanned: {p.name}")
        text = get_file_content(p.as_posix())
        if not text:
            continue
        rel_path = p.relative_to(Path(codebase_root)).as_posix()
        docs.append(
            Document(
                page_content= rel_path + text,
                metadata={
                    "doc_type": "file",
                    "path": rel_path,
                    "id": make_id(text),
                },
            )
        )

    # GitHub issues / PRs from jsonl (optional)
    if issues_jsonl_path and Path(issues_jsonl_path).exists():
        with open(issues_jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    issue = json.loads(line)
                except json.JSONDecodeError as e:
                    raise e

                issue_text_parts = [
                    f"{issue.get('title')}\n",
                    f"{issue.get('body') or ''}\n"
                ]
                
                discussion_summary = issue.get('discussion_summary')
                if discussion_summary:
                    issue_text_parts.append(f"\n{discussion_summary}\n")
                
                issue_text = "".join(issue_text_parts)
                
                log_and_print(f"Processing issue #{issue.get('number')}: {issue.get('title', '')[:50]}...")
                docs.append(
                    Document(
                        page_content=issue_text,
                        metadata={
                            "full_content": issue_text,
                            "doc_type": "issue",
                            "repo": issue.get("repo"),
                            "number": issue.get("number"),
                            "url": issue.get("html_url"),
                            "has_summary": bool(discussion_summary),
                            "id": make_id(issue_text),
                        },
                    )
                )
    
    log_and_print(f"Total documents BEFORE splitting: {len(docs)}")
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=splitter_chunk_size, chunk_overlap=50)
    chunked_docs = text_splitter.split_documents(docs)
    log_and_print(f"Total documents AFTER splitting: {len(chunked_docs)}")
    
    log_and_print(f"######### Calculating Hashes....")
    ids = [f"{make_id(chunk.page_content)}_{i}" for i, chunk in enumerate(chunked_docs)]
    # for chunk in chunked_docs:
    #     log_and_print(f"Chunk ID: {make_id(chunk.page_content)} | \nContent Preview: {chunk.page_content[:100]}---")

    log_and_print(f"######### Adding documents to vector store....")
    for i in range(0, len(chunked_docs), batch_size):
        batch_docs = chunked_docs[i:i + batch_size]
        batch_ids = ids[i:i + batch_size]
        log_and_print(f"Adding batch {i // batch_size + 1}/{(len(chunked_docs) + batch_size - 1) // batch_size}: documents {i} to {i + len(batch_docs)}")
        vector_store_from_client.add_documents(documents=batch_docs, ids=batch_ids)
    
    return vector_store_from_client


## 6. guthub_utils.py

In [11]:

# ---------- helpers ----------


def _parse_repo_url(repo_url: str) -> Tuple[str, str]:
    """
    Args:
        repo_url: String
        Examples:
            - "https://github.com/OWNER/REPO"
            - "https://github.com/OWNER/REPO.git"
            - "http(s)://github.com/OWNER/REPO/anything..."
            - "git@github.com:OWNER/REPO.git"
    Returns:
        (owner, repo)
    """
    repo_url = repo_url.strip()

    # git@github.com:owner/repo.git
    m = re.match(
        r"^git@github\.com:(?P<owner>[^/]+)/(?P<repo>[^/]+?)(?:\.git)?$", repo_url
    )
    if m:
        return m.group("owner"), m.group("repo")

    # https://github.com/owner/repo(.git)?/...
    parsed = urlparse(repo_url)
    if parsed.netloc.lower() != "github.com":
        raise ValueError(f"Not a GitHub URL: {repo_url}")

    parts = [p for p in parsed.path.split("/") if p]
    if len(parts) < 2:
        raise ValueError(f"Could not parse owner/repo from URL: {repo_url}")

    owner, repo = parts[0], parts[1]
    repo = re.sub(r"\.git$", "", repo, flags=re.I)
    return owner, repo


def _make_github(token: str = "") -> Github:
    if "GITHUB_TOKEN" not in os.environ and not token:
        token = getpass.getpass("Enter your GITHUB_TOKEN: ")

    return Github(auth=Auth.Token(os.getenv("GITHUB_TOKEN", token)),
                  per_page=100,
                  seconds_between_requests=0.25,
                  seconds_between_writes=1.0,
                  retry=GithubRetry(),
        )


def _ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def _write_jsonl(path: str, records, mode: str = "w") -> None:
    _ensure_dir(os.path.dirname(path))
    with open(path, mode, encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")


def _append_jsonl(path: str, record) -> None:
    """Append a single record to a JSONL file (thread-safe with file locking)."""
    _ensure_dir(os.path.dirname(path))
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def _read_processed_issue_numbers(issues_path: str) -> Set[int]:
    """Read all issue numbers that have already been processed from a JSONL file."""
    processed = set()
    if not os.path.exists(issues_path):
        return processed
    
    with open(issues_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                if "number" in rec:
                    processed.add(int(rec["number"]))
            except json.JSONDecodeError:
                continue
    return processed


def _read_processed_comment_ids(comments_path: str) -> Set[int]:
    """Read all comment IDs that have already been processed from a JSONL file."""
    processed = set()
    if not os.path.exists(comments_path):
        return processed
    
    with open(comments_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                if "comment_id" in rec:
                    processed.add(int(rec["comment_id"]))
            except json.JSONDecodeError:
                continue
    return processed


def _is_pull_request(issue) -> bool:
    return getattr(issue, "pull_request", None) is not None




def _try_git_checkout(out_dir: str, commit_hash: str) -> bool:
    """
    Attempts to checkout the specified commit hash in the given directory.
    
    Args:
        out_dir: Directory that may contain a git repository
        commit_hash: The commit hash to checkout
        
    Returns:
        True if checkout was successful.
        Otherwise, Returns False.
    """
    if not os.path.isdir(out_dir):
        return False
    
    git_dir = os.path.join(out_dir, ".git")
    if not os.path.isdir(git_dir):
        return False
    
    try:
        subprocess.run(
            ["git", "fetch", "--all", "--progress", "--keep"],
            cwd=out_dir,
            check=True,
            capture_output=True,
            text=True,
        )
        
        subprocess.run(
            ["git", "checkout", commit_hash],
            cwd=out_dir,
            check=True,
            capture_output=True,
            text=True,
        )
        print(f"✓ Successfully checked out to commit {commit_hash[:7]} in existing repo")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Could not checkout {commit_hash[:7]}: {e.stderr.strip() if e.stderr else str(e)}")
        return False
    except Exception as e:
        print(f"Git checkout failed: {e}")
        return False


def _init_git_repo_with_upstream(
    out_dir: str,
    repo_url: str,
    token: str = "",
) -> bool:
    """
    Initializes a git repository in out_dir and sets up the upstream remote.
    
    Args:
        out_dir: Directory to initialize as a git repo
        repo_url: GitHub repo URL to set as upstream (origin)
        token: GitHub token for authenticated access
        
    Returns:
        True if successful, False otherwise
    """
    owner, name = _parse_repo_url(repo_url)
    tok = token or os.getenv("GITHUB_TOKEN")
    
    if tok:
        remote_url = f"https://{tok}@github.com/{owner}/{name}.git"
    else:
        remote_url = f"https://github.com/{owner}/{name}.git"
    
    try:
        subprocess.run(
            ["git", "init"],
            cwd=out_dir,
            check=True,
            capture_output=True,
            text=True,
        )
        print(f"✓ Initialized git repository in {out_dir}")
        
        subprocess.run(
            ["git", "remote", "add", "origin", remote_url],
            cwd=out_dir,
            check=True,
            capture_output=True,
            text=True,
        )
        print(f"✓ Added origin remote: https://github.com/{owner}/{name}.git")
        
        subprocess.run(
            ["git", "fetch", "--all", "--progress", "--keep"],
            cwd=out_dir,
            check=True,
            capture_output=True,
            text=True,
        )
        print("✓ Fetched all refs from origin")
        
        return True
        
    except subprocess.CalledProcessError as e:
        print(f"✗ Failed to initialize git repo: {e.stderr.strip() if e.stderr else str(e)}")
        return False
    except Exception as e:
        print(f"✗ Error initializing git repo: {e}")
        return False

# ---------- public functions ----------

def download_codebase(
    repo_url: str,
    ref: Optional[str] = None,
    token: str = "",
):
    """
    Downloads the repository contents as raw files into `projects/`, preserving the folder structure.

    Args:
        repo_url: GitHub repo URL (https or git@)
        ref: branch/tag/commit (default: repo.default_branch)
        token: GitHub token (alternatively set GITHUB_TOKEN env var)
    """
    g = _make_github(token)
    owner, name = _parse_repo_url(repo_url)
    
    repo = g.get_repo(f"{owner}/{name}")
    if ref is None:
        ref = repo.default_branch
    name = "projects/" + name.lower()
    sleep(1.0)
    if _try_git_checkout(name, ref):
        return True

    tok = token or os.getenv("GITHUB_TOKEN")

    _ensure_dir(name)
    _init_git_repo_with_upstream(name, repo_url, token=tok)
    _try_git_checkout(name, ref)


def save_issues(
    repo_url: str,
    out_dir: str = "issues",
    filename: str = "issues.jsonl",
    token: str = "",
    include_prs: bool = False,
):
    """
    Saves repository issues (title + body + metadata) as JSON Lines.

    Args:
        repo_url: GitHub repo URL
        out_dir: output directory (default: "issues")
        filename: output file name (default: "issues.jsonl")
        token: GitHub token or env var
        include_prs: include PRs in addition to issues (default: False)
    """
    g = _make_github(token)
    owner, name = _parse_repo_url(repo_url)
    repo = g.get_repo(f"{owner}/{name}")

    print("====== Adding issues... ======")
    sleep(1.0)

    records = []
    for issue in repo.get_issues(state="all"):  # PyGithub handles pagination
        if not include_prs and _is_pull_request(issue):
            continue

        labels = (
            [lbl.name for lbl in issue.get_labels()]
            if hasattr(issue, "get_labels")
            else []
        )
        assignees = [a.login for a in (issue.assignees or [])]

        print("=========")
        print(f"number: {issue.number}")
        print(f"title: {issue.title}")
        print(f"is_pull_request: {_is_pull_request(issue)}")
        print("==============")

        records.append(
            {
                "repo": f"{owner}/{name}",
                "number": issue.number,
                "title": issue.title or "",
                "body": issue.body or "",
                "state": issue.state,
                "created_at": issue.created_at,
                "updated_at": issue.updated_at,
                "closed_at": issue.closed_at,
                "user": getattr(issue.user, "login", None),
                "assignees": assignees,
                "labels": labels,
                "is_pull_request": _is_pull_request(issue),
                "html_url": issue.html_url,
                "comments_count": issue.comments,
            }
        )

    _write_jsonl(os.path.join(out_dir, filename), records)


def save_issue_comments(
    repo_url: str,
    out_dir: str = "comments",
    filename: str = "comments.jsonl",
    token: str = "",
    include_prs: bool = False,
):
    """
    Saves all issue comments (including comments on PR threads as "issue comments")
    as JSON Lines.

    Args:
        repo_url: GitHub repo URL
        out_dir: output directory (default: "comments")
        filename: output file name (default: "comments.jsonl")
        token: GitHub token or env var
        include_prs: include PR issues when iterating (default: False)
    """
    g = _make_github(token)
    owner, name = _parse_repo_url(repo_url)
    repo = g.get_repo(f"{owner}/{name}")

    print("====== Adding issues' comments... ======")
    sleep(1.0)

    records = []
    for issue in repo.get_issues(state="all"):
        if not include_prs and _is_pull_request(issue):
            continue
        for c in issue.get_comments():
            print("===")
            print(f"number: {c.id}")
            print(f"body: {c.body}")
            print("=======")
            records.append(
                {
                    "repo": f"{owner}/{name}",
                    "issue_number": issue.number,
                    "issue_title": issue.title or "",
                    "comment_id": c.id,
                    "user": getattr(c.user, "login", None),
                    "body": c.body or "",
                    "created_at": c.created_at,
                    "updated_at": c.updated_at,
                    "html_url": c.html_url,
                }
            )

    _write_jsonl(os.path.join(out_dir, filename), records)


def get_commit_date_posix(repo_url, commit_hash):
    g = _make_github()
    owner, name = _parse_repo_url(repo_url)
    repo = g.get_repo(f"{owner}/{name}")

    commit = repo.get_commit(commit_hash)
    cutoff_date = commit.commit.author.date.timestamp()
    return str(cutoff_date)


def save_issues_and_comments_before_commit(
    repo_url: str,
    commit_hash: str,
    issues_out_dir: str = "issues",
    issues_filename: str = "issues_before_commit.jsonl",
    comments_out_dir: str = "comments",
    comments_filename: str = "comments_before_commit.jsonl",
    token: str = "",
    include_prs: bool = True,
    max_workers: int = 3,
    time_duration: int = 180,
    save_comments: bool = False,
    only_closed: bool = False
):
    """
    Saves repository issues and comments that were created before a given commit.
    The commit's author date is used as the cutoff time.

    Args:
        repo_url: GitHub repo URL
        commit_hash: Git commit hash to use as the time cutoff
        issues_out_dir: output directory for issues (default: "issues")
        issues_filename: output file name for issues (default: "issues_before_commit.jsonl")
        comments_out_dir: output directory for comments (default: "comments")
        comments_filename: output file name for comments (default: "comments_before_commit.jsonl")
        token: GitHub token or env var
        include_prs: include PRs in addition to issues (default: False)
        max_workers: number of threads for parallel processing (default: 3)
        time_duration: since this time before commit author date [days](180)
        save_comments: save comments or not [False]
        only_closed: to save only closed issues or not [False]
    """

    g = _make_github(token)
    owner, name = _parse_repo_url(repo_url)
    repo = g.get_repo(f"{owner}/{name}")

    _ensure_dir("results")
    print(f"====== Fetching commit {commit_hash[:7]} ======")
    sleep(1.0)
    
    commit = repo.get_commit(commit_hash)
    cutoff_date = commit.commit.author.date
    earliest_date = cutoff_date - timedelta(days=time_duration)
    print(f"Commit date: {cutoff_date}")
    print(f"Earliest date (6 months back): {earliest_date}")
    print(f"Filtering issues and comments created between {earliest_date} and {cutoff_date}...")
    
    issues_filename = issues_filename + str(cutoff_date.timestamp()) + ".jsonl"
    comments_filename = comments_filename + str(cutoff_date.timestamp()) + ".jsonl"
    issues_path = os.path.join(issues_out_dir, issues_filename)
    comments_path = os.path.join(comments_out_dir, comments_filename)
    
    processed_issues = _read_processed_issue_numbers(issues_path)
    processed_comments = _read_processed_comment_ids(comments_path)
    
    if processed_issues:
        print(f"Found {len(processed_issues)} already processed issues, will skip them.")
    if processed_comments:
        print(f"Found {len(processed_comments)} already processed comments, will skip them.")

    print("\n====== Collecting issues before commit... ======")
    sleep(1.0)
    
    issues_lock = threading.Lock()
    comments_lock = threading.Lock()
    stats = {"issues_added": 0, "issues_skipped": 0, "comments_added": 0, "comments_skipped": 0}
    stats_lock = threading.Lock()
    
    def process_issue(issue: Issue):
        """Process a single issue and its comments. Returns counts of processed items."""
        local_stats = {"issues_added": 0, "issues_skipped": 0, "comments_added": 0, "comments_skipped": 0}
        
        if not include_prs and _is_pull_request(issue):
            return local_stats
        
        if only_closed and issue.state == "open":
            return local_stats

        if issue.created_at > cutoff_date:
            return local_stats
        
        if issue.created_at < earliest_date:
            return local_stats
        
        issue_number = issue.number
        
        if issue_number not in processed_issues:
            labels = (
                [lbl.name for lbl in issue.get_labels()]
                if hasattr(issue, "get_labels")
                else []
            )
            assignees = [a.login for a in (issue.assignees or [])]
            
            issue_record = {
                "repo": f"{owner}/{name}",
                "number": issue_number,
                "title": issue.title or "",
                "body": issue.body or "",
                "state": issue.state,
                "created_at": issue.created_at,
                "updated_at": issue.updated_at,
                "closed_at": issue.closed_at,
                "user": getattr(issue.user, "login", None),
                "assignees": assignees,
                "labels": labels,
                "is_pull_request": _is_pull_request(issue),
                "html_url": issue.html_url,
                "comments_count": issue.comments,
            }
            
            with issues_lock:
                _append_jsonl(issues_path, issue_record)
                processed_issues.add(issue_number)
    
            print(f"✓ Issue #{issue_number}: {issue.title} (created: {issue.created_at})")
            local_stats["issues_added"] = 1
        else:
            print(f"⏭ Skipping issue #{issue_number} (already processed)")
            local_stats["issues_skipped"] = 1

        if not save_comments:
            return local_stats

        try:
            for c in issue.get_comments():
                if c.created_at > cutoff_date:
                    continue
                    
                comment_id = c.id
                
                if comment_id not in processed_comments:
                    comment_record = {
                        "repo": f"{owner}/{name}",
                        "issue_number": issue_number,
                        "issue_title": issue.title or "",
                        "comment_id": comment_id,
                        "user": getattr(c.user, "login", None),
                        "body": c.body or "",
                        "created_at": c.created_at,
                        "updated_at": c.updated_at,
                        "html_url": c.html_url,
                    }
                    
                    with comments_lock:
                        _append_jsonl(comments_path, comment_record)
                        processed_comments.add(comment_id)  # Update set to avoid duplicates
                    
                    print(f"  ✓ Comment #{comment_id} on issue #{issue_number}")
                    local_stats["comments_added"] += 1
                else:
                    local_stats["comments_skipped"] += 1
        except Exception as e:
            print(f"  ⚠ Error fetching comments for issue #{issue_number}: {e}")
        
        return local_stats
    
    print("Searching for issues within date range using GitHub Search API...")
    
    earliest_str = earliest_date.strftime("%Y-%m-%d")
    cutoff_str = cutoff_date.strftime("%Y-%m-%d")
    
    search_query = f"repo:{owner}/{name} is:issue created:{earliest_str}..{cutoff_str}"
    if include_prs:
        search_query = f"repo:{owner}/{name} created:{earliest_str}..{cutoff_str}"
    
    print(f"Search query: {search_query}")
    search_results = g.search_issues(search_query)
    
    issues_to_process = []
    for issue in search_results:
        if issue.number not in processed_issues:
            issues_to_process.append(issue.number)
    
    print(f"Found {len(issues_to_process)} issues to process (after filtering already processed)\n")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_issue, repo.get_issue(issue_num)): issue_num for issue_num in issues_to_process}
        
        for future in as_completed(futures):
            try:
                local_stats = future.result()
                with stats_lock:
                    for key in stats:
                        stats[key] += local_stats[key]
            except Exception as e:
                issue_id = futures[future]
                print(f"⚠ Error processing issue #{issue_id}: {e}")
    
    print(f"\n" + "=" * 50)
    print(f"✓ Added {stats['issues_added']} new issues (skipped {stats['issues_skipped']} existing)")
    print(f"✓ Added {stats['comments_added']} new comments (skipped {stats['comments_skipped']} existing)")
    print(f"Issues saved to: {issues_path}")
    print(f"Comments saved to: {comments_path}")
    
    return (issues_filename, comments_filename)


## 7. process_layer.py

In [12]:
# LLM = None

if not LLM:
    print("""
          1- qwen2.5-coder:0.5b (default)
          2- gemma2:2b
          3- gemini-2.5-flash
          4- qwen3:14b
          5- deepseek-r1:14b
          6- gemma3:12b
          7- upstage/solar-pro-3:free
          8- liquid/lfm-2.5-1.2b-thinking:free
          9- tngtech/deepseek-r1t2-chimera:free
          10- arcee-ai/trinity-large-preview:free
          """)
    selected = input("Select a model to continue: ")
    # selected = "2"
    if selected == "1":
        log_and_print("qwen2.5-coder:0.5b selected")
        LLM = ChatOllama(model="qwen2.5-coder:0.5b", temperature=0, top_p=0.8)
    elif selected == "2":
        log_and_print("gemma2:2b selected")
        LLM = ChatOllama(model="gemma2:2b", temperature=0, top_p=0.8)
    elif selected == "3":
        log_and_print("gemini-2.5-flash")
        if "GOOGLE_API_KEY" not in os.environ:
            os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your GOOGLE_API_KEY: ")

        LLM = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            temperature=0,
            top_p=0.8,
            max_retries=1,
            timeout=180,
        )
    elif selected == "4":
        log_and_print("qwen3:14b selected")
        LLM = ChatOllama(model="qwen3:14b", temperature=0, top_p=0.8)
    elif selected == "5":
        log_and_print("deepseek-r1:14b selected")
        LLM = ChatOllama(model="deepseek-r1:14b", temperature=0, top_p=0.8)
    elif selected == "6":
        log_and_print("gemma3:12b selected")
        LLM = ChatOllama(model="gemma3:12b", temperature=0, top_p=0.8)
    elif selected == "7":
        if not os.environ.get("OPENROUTER_API_KEY"):
            os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OPENROUTER_API_KEY: ")

        LLM = ChatOpenAI(
            api_key=os.environ.get("OPENROUTER_API_KEY"),
            base_url="https://openrouter.ai/api/v1",
            model="upstage/solar-pro-3:free",
        )
    elif selected == "8":
        if not os.environ.get("OPENROUTER_API_KEY"):
            os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OPENROUTER_API_KEY: ")

        LLM = ChatOpenAI(
            api_key=os.environ.get("OPENROUTER_API_KEY"),
            base_url="https://openrouter.ai/api/v1",
            model="liquid/lfm-2.5-1.2b-thinking:free",
        )
    elif selected == "9":
        if not os.environ.get("OPENROUTER_API_KEY"):
            os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OPENROUTER_API_KEY: ")

        LLM = ChatOpenAI(
            api_key=os.environ.get("OPENROUTER_API_KEY"),
            base_url="https://openrouter.ai/api/v1",
            model="tngtech/deepseek-r1t2-chimera:free",
        )
    elif selected == "10":
        if not os.environ.get("OPENROUTER_API_KEY"):
            os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OPENROUTER_API_KEY: ")

        LLM = ChatOpenAI(
            api_key=os.environ.get("OPENROUTER_API_KEY"),
            base_url="https://openrouter.ai/api/v1",
            model="arcee-ai/trinity-large-preview:free",
        )
    else:
        LLM = ChatOllama(model="qwen2.5-coder:0.5b", temperature=0)


def context_retriever(project_path: str, issues_path: str, query_content: str, top_k: int = 3, with_summary: bool = False, with_tree: bool = True):
    vector_store = build_vector_store(project_path, issues_path)

    log_and_print(f"Querying for similar docs...\n\nQuery Text:\n{query_content}\n")
    similar_docs = vector_store.similarity_search(
        query_content,
        k = top_k * 2 + 1,
    )
    # retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 2 * (top_k + 1), "fetch_k": 40, "lambda_mult": 0.25})
    
    # similar_docs = retriever.invoke(query_content)
    log_and_print(f"#{len(similar_docs)} Similar docs found:\n\n")
    related_files_block = ""
    related_issues_block = ""
    found = set()
    counter = 0
    for d in similar_docs:
        log_and_print(f"############# Doc Path: #############")
        log_and_print(f"{d.metadata.get("path")}")
        log_and_print(f"{d.metadata.get("url")}")
        log_and_print(f"{top_k}")
        log_and_print(f"{counter}")
        if d.metadata.get("id") in found:
            log_and_print(f"!!doc already checked in {found} | skipping...")
            continue
        found.add(d.metadata.get("id"))

        if counter >= top_k + 1:
            log_and_print(f"!!!!Breaking...")
            break
        if d.metadata.get("doc_type") == "file":
            log_and_print(f"Not The same file {d.metadata.get("id") != make_id(query_content)}")
            c: str = get_file_content(f"{project_path}/{d.metadata.get("path")}")
            related_files_block += "".join(
                f"---BEGIN RELATED FILE (Line Numbered)---\n"
                f"FILEPATH: ./{d.metadata.get('path')}\n"
                f"CONTENT:\n{add_line_numbers(c[:4000].rstrip())}\n"
                f"---END RELATED FILE---\n"
            )
            counter += 1
        elif d.metadata.get("doc_type") == "issue" and d.metadata.get("full_content") != query_content:
            c: str = d.metadata.get("full_content")
            log_and_print(f"Not The same issue {d.metadata.get("full_content") != query_content}")
            related_issues_block += "".join(
                f"---BEGIN RELATED ISSUE: {d.metadata.get('repo')}#{d.metadata.get('number')}---\n"
                f"CONTENT:\n{c[:4000].rstrip()}\n"
                f"---END RELATED ISSUE---\n"
            )
            counter += 1


    context_payload = (
        "[TOP-" + str(top_k) + " RELEVANT CONTEXT as FILES or ISSUES]\n"
        + (related_files_block)
        + (related_issues_block)
        + "\n[END RELEVANT CONTEXT]\n"
        + f"[PROJECT TREE]\n{get_project_tree(project_path)}\n[END PROJECT TREE]\n" if with_tree else ""
    )
    return context_payload


def context_retriever_both(project_path: str, issues_path: str, query_content: str, top_k: int = 3):
    vector_store = build_vector_store(project_path, issues_path)

    log_and_print(f"Querying for similar docs...\n\nQuery Text:\n{query_content}\n")
    similar_docs = vector_store.similarity_search(
        query_content,
        k = 2 * (top_k + 1),
    )
    retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": top_k + 1, "lambda_mult": 0.5, "filter": {"doc_type": "file"}})
    
    similar_docs = retriever.invoke(query_content)

    log_and_print(f"#{top_k} Similar docs found:\n")
    related_files_block = ""
    related_issues_block = ""
    found = set()
    file_counter = 0
    issue_counter = 0
    for d in similar_docs:
        log_and_print(f"{d.metadata}\n")
        if d.metadata.get("id") in found:
            continue
        log_and_print(f"############# Doc Path: #############")
        log_and_print(f"{d.metadata.get("path")}")

        found.add(d.metadata.get("id"))
        if d.metadata.get("doc_type") == "file" and d.page_content != query_content and file_counter < top_k:
            c: str = get_file_content(f"{project_path}/{d.metadata.get("path")}")
            related_files_block += "".join(
                f"---BEGIN RELATED FILE (Line Numbered)---\n"
                f"FILEPATH: ./{d.metadata.get('path')}\n"
                f"CONTENT:\n{add_line_numbers(c[:4000].rstrip())}\n"
                f"---END RELATED FILE---\n"
            )
            file_counter += 1

    for d in similar_docs:
        if d.metadata.get("id") in found:
            continue
        log_and_print(f"############# ISSUE URL: #############")
        log_and_print(f"{d.metadata.get("html_url")}")
        found.add(d.metadata.get("id"))
        if d.metadata.get("doc_type") == "issue" and make_id(query_content) != d.metadata.get("id") and issue_counter < top_k:
            c: str = d.metadata.get("full_content")
            related_issues_block += "".join(
                f"---BEGIN RELATED ISSUE: {d.metadata.get('repo')}#{d.metadata.get('number')}---\n"
                f"CONTENT:\n{c[:4000].rstrip()}\n"
                f"---END RELATED ISSUE---\n"
            )
            issue_counter += 1

    context_payload = (
        "[TOP-" + str(top_k) + " RELEVANT CONTEXT as FILES]\n"
        + (related_files_block)
        + "\n[END RELEVANT FILES]\n"
        + "[TOP-" + str(top_k) + " RELEVANT CONTEXT as ISSUES]\n"
        + (related_issues_block)
        + "\n[END RELEVANT ISSUES]\n"
    )
    return context_payload


def retriever_bm25_docs(project_path: str, issues_path: str, query_content: str, top_k: int = 3, with_summary: bool = False, with_tree: bool = True) -> dict:
    """
    Retrieve top-30 files using BM25 retriever and return their documentation.
    
    Args:
        project_path: Path to the project root
        issues_path: Path to issues (unused but kept for API compatibility)
        query_content: Query text to search for
        top_k: Unused but kept for API compatibility
        with_summary: Unused but kept for API compatibility
        with_tree: Unused but kept for API compatibility
        
    Returns:
        Dict with file paths as keys and file documentation dicts as values
    """
    # Build document list from Python files in the project
    file_docs = []
    for file_path in iter_text_files(project_path):
        try:
            content = file_path.read_text(encoding="utf-8")
            rel_path = file_path.relative_to(project_path).as_posix()
            file_docs.append(Document(
                page_content=content,
                metadata={"path": rel_path}
            ))
        except Exception as e:
            log_and_print(f"[{type(e).__name__}] Failed to read {file_path}: {e}")
    
    if not file_docs:
        return {}
    
    log_and_print(f"Found {len(file_docs)} Python files in {project_path}")
    
    # Create BM25 retriever from file documents
    bm25_retriever = BM25Retriever.from_documents(file_docs, k=30)
    
    log_and_print(f"BM25 retrieving top-30 files for query...\n")
    retrieved_docs = bm25_retriever.invoke(query_content)
    
    # Build result dict with file path -> file documentation
    result = {}
    for doc in retrieved_docs:
        file_path = doc.metadata.get("path")
        if file_path:
            try:
                file_doc = extract_file_documentation(doc.page_content)
                result[file_path] = file_doc
                log_and_print(f"Extracted documentation for: {file_path}")
            except Exception as e:
                log_and_print(f"[{type(e).__name__}] Failed to extract documentation for {file_path}: {e}")
    
    return result


def build_context_payload_from_docs(file_docs: dict) -> str:
    """
    Build a JSON context payload for LLM input from file documentation dict.
    
    Args:
        file_docs: Dict with file paths as keys and file documentation dicts as values
                   (output from retriever_bm25_docs)
        
    Returns:
        JSON string containing the context payload for LLM input
    """
    
    payload = {
        "retrieved_file_documentations": [
            {
                "path": file_path,
                "documentation": doc
            }
            for file_path, doc in file_docs.items()
        ]
    }

    return json.dumps(payload, indent=2, ensure_ascii=False)


def add_line_numbers(text: str) -> str:
    lines = text.splitlines()
    return "\n".join(f"{i:04d}: {line}" for i, line in enumerate(lines, start=1))


## 8. apply_patch.py

In [13]:
def validate_patch(patch: str) -> bool:
    """
    Basic validation to check if the patch string looks like a unified diff.
    This is a heuristic check and not a full proof validation.

    patch example:
diff --git a/file.py b/file.py
--- a/file.py
+++ b/file.py
@@ -1,27 +1,35 @@
 def euclidean(a, b):
-    while b:
-        a, b = b, a % b
-    return a
+    if b == 0:
+        return a
+    return euclidean(b, a % b)
 
 
 def bresenham(x0, y0, x1, y1):
     points = []
     dx = abs(x1 - x0)
     dy = abs(y1 - y0)
-    sx = 1 if x0 < x1 else -1
-    sy = 1 if y0 < y1 else -1
-    err = dx - dy
+    x, y = x0, y0
+    sx = -1 if x0 > x1 else 1
+    sy = -1 if y0 > y1 else 1
 
-    while True:
-        points.append((x0, y0))
-        if x0 == x1 and y0 == y1:
-            break
-        e2 = 2 * err
-        if e2 > -dy:
+    if dx > dy:
+        err = dx / 2.0
+        while x != x1:
+            points.append((x, y))
             err -= dy
-            x0 += sx
-        if e2 < dx:
-            err += dx
-            y0 += sy
+            if err < 0:
+                y += sy
+                err += dx
+            x += sx
+    else:
+        err = dy / 2.0
+        while y != y1:
+            points.append((x, y))
+            err -= dx
+            if err < 0:
+                x += sx
+                err += dy
+            y += sy
 
+    points.append((x, y))
     return points
    """
    
    if not patch or not isinstance(patch, str):
        log_and_print("[ValidationError] Patch is empty or not a string")
        return False
    
    lines = patch.strip().splitlines()
    if not lines:
        log_and_print("[ValidationError] Patch contains no lines")
        return False
    
    # Check for diff header (diff --git or ---)
    has_diff_header = any(
        line.strip().startswith("diff --git") or line.strip().startswith("---") 
        for line in lines[:5]
    )
    if not has_diff_header:
        log_and_print("[ValidationError] Missing diff header (diff --git or ---)")
        return False
    
    # Check for hunk header (@@ ... @@)
    hunk_pattern = re.compile(r'^@@\s+-\d+(?:,\d+)?\s+\+\d+(?:,\d+)?\s+@@')
    has_hunk = any(hunk_pattern.match(line.strip()) for line in lines)
    if not has_hunk:
        log_and_print("[ValidationError] Missing hunk header (@@ -x,y +x,y @@)")
        return False
    
    # Check that there are actual diff lines (starting with +, -, or space)
    diff_line_pattern = re.compile(r'^[ +\-]')
    has_diff_lines = False
    in_hunk = False
    for line in lines:
        if hunk_pattern.match(line.strip()):
            in_hunk = True
            continue
        if in_hunk and diff_line_pattern.match(line.strip()):
            has_diff_lines = True
            break
    
    if not has_diff_lines:
        log_and_print("[ValidationError] No diff content lines found (lines starting with +, -, or space)")
        return False
    
    return True


def apply_patch_and_get_diff(patch: PatchSuggestions, file_path: str) -> str:
    """
    Apply PatchSuggestions to a file and return the git-style unified diff.
    
    Args:
        patch: PatchSuggestions containing edits to apply
        file_path: Path to the file to patch
        
    Returns:
        Git-style unified diff string
        
    Raises:
        ValueError: If snippets don't match, line numbers are invalid, or edits overlap
        FileNotFoundError: If file doesn't exist
    """
    # Read the original file content
    if len(patch.edits) == 0:
        log_and_print("$ NO REPONSE...........")
    original_content = get_file_content(file_path)

    log_and_print(f">>>>>>>> Local file content:\n{original_content}\n<<<<<<<<<<<")
    # Split into lines (keeping line endings for accurate reconstruction)
    original_lines = original_content.splitlines(keepends=True)
    # Ensure last line has newline for consistency
    if original_lines and not original_lines[-1].endswith('\n'):
        original_lines[-1] += '\n'
    
    # Validate and collect edits with their line ranges
    edits_with_ranges: List[tuple] = []  # (start, end, snippet)
    
    for edit in patch.edits:
        # Convert 1-indexed inclusive to 0-indexed (start is 0-indexed, end is exclusive)
        start = edit.line_start_for_editing - 1
        end = edit.line_end_for_editing  # Already exclusive after -1 +1
        
        # Validate line numbers are within bounds
        if start < 0:
            raise ValueError(f"Invalid start line {edit.line_start_for_editing}: must be >= 1")
        if end > len(original_lines):
            raise ValueError(
                f"Invalid end line {edit.line_end_for_editing}: file only has {len(original_lines)} lines"
            )
        if start >= end:
            raise ValueError(
                f"Invalid line range: start ({edit.line_start_for_editing}) must be < end ({edit.line_end_for_editing})"
            )
        
        # Extract the actual content at those lines
        actual_snippet = "".join(original_lines[start:end])
        expected_snippet = edit.exact_existing_buggy_snippet
        
        # Normalize for comparison (handle trailing newline differences)
        actual_normalized = actual_snippet.rstrip('\n')
        expected_normalized = expected_snippet.rstrip('\n')
        
        if actual_normalized != expected_normalized:
            raise ValueError(
                f"Snippet mismatch at lines {edit.line_start_for_editing}-{edit.line_end_for_editing}.\n"
                f"Expected:\n{repr(expected_snippet)}\n"
                f"Actual:\n{repr(actual_snippet)}"
            )
        
        edits_with_ranges.append((start, end, edit))
    
    # Sort by start line to check for overlaps
    edits_with_ranges.sort(key=lambda x: x[0])
    
    # Check for overlapping edits
    for i in range(len(edits_with_ranges) - 1):
        current_start, current_end, current_edit = edits_with_ranges[i]
        next_start, next_end, next_edit = edits_with_ranges[i + 1]
        
        if current_end > next_start:
            raise ValueError(
                f"Overlapping edits detected:\n"
                f"Edit 1: lines {current_edit.line_start_for_editing}-{current_edit.line_end_for_editing}\n"
                f"Edit 2: lines {next_edit.line_start_for_editing}-{next_edit.line_end_for_editing}"
            )
    
    # Apply edits in reverse order (from end of file to beginning)
    # This preserves line numbers for earlier edits
    edits_with_ranges.sort(key=lambda x: x[0], reverse=True)
    
    patched_lines = original_lines.copy()
    
    for start, end, edit in edits_with_ranges:
        replacement = edit.correct_replacement_snippet
        
        # Ensure replacement ends with newline if original block did
        original_block = "".join(original_lines[start:end])
        if original_block.endswith('\n') and not replacement.endswith('\n'):
            replacement += '\n'
        
        # Split replacement into lines
        replacement_lines = replacement.splitlines(keepends=True)
        if replacement_lines and not replacement_lines[-1].endswith('\n'):
            replacement_lines[-1] += '\n'
        
        # Replace the lines
        patched_lines[start:end] = replacement_lines
    
    # Reconstruct patched content
    patched_content = "".join(patched_lines)
    
    # Generate unified diff
    original_for_diff = original_content.splitlines(keepends=True)
    patched_for_diff = patched_content.splitlines(keepends=True)
    
    diff_lines = list(unified_diff(
        original_for_diff,
        patched_for_diff,
        fromfile=f"a/{patch.file_path_to_edit}",
        tofile=f"b/{patch.file_path_to_edit}",
        lineterm=""
    ))
    
    log_and_print(f">>>>>>>> model patch content:\n{patched_for_diff}\n<<<<<<<<<<")
    log_and_print(f"Model suggested path: {patch.file_path_to_edit}")

    # Build git-style diff header
    if not diff_lines:
        diff_output = f"diff --git a/{patch.file_path_to_edit} b/{patch.file_path_to_edit}\n"
    else:
        diff_output = f"diff --git a/{patch.file_path_to_edit} b/{patch.file_path_to_edit}\n"
        diff_output += "\n".join(diff_lines)
        if not diff_output.endswith('\n'):
            diff_output += '\n'
    
    log_and_print(f"#####@@@@@@##### DIFF VIEW #####@@@@@@#####\n{diff_output}")

    return diff_output


def extract_file_documentation(file_content: str) -> dict:
    """
    Extract documentation from a Python file content string.
    
    Args:
        file_content: The content of a Python file as a string.
        
    Returns:
        A dictionary containing:
        - module_docstring: The module-level docstring (if available)
        - classes: List of class info with name, docstring, and method signatures
        - functions: List of function info with name and first/last 5 lines
    """
    import ast
    
    result = {
        "module_docstring": None,
        "classes": [],
        "functions": []
    }
    
    try:
        tree = ast.parse(file_content)
    except SyntaxError:
        return result
    
    lines = file_content.splitlines()
    
    # Extract module docstring
    result["module_docstring"] = ast.get_docstring(tree)
    
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, ast.ClassDef):
            # Extract class information
            class_info = {
                "name": node.name,
                "docstring": ast.get_docstring(node),
                "methods": []
            }
            
            # Add base classes to name if present
            if node.bases:
                base_names = []
                for base in node.bases:
                    if isinstance(base, ast.Name):
                        base_names.append(base.id)
                    elif isinstance(base, ast.Attribute):
                        base_names.append(ast.unparse(base))
                    else:
                        base_names.append(ast.unparse(base))
                class_info["name"] = f"{node.name}({', '.join(base_names)})"
            
            # Extract method signatures
            for item in node.body:
                if isinstance(item, ast.FunctionDef) or isinstance(item, ast.AsyncFunctionDef):
                    # Build method signature
                    args = []
                    for arg in item.args.args:
                        arg_str = arg.arg
                        if arg.annotation:
                            arg_str += f": {ast.unparse(arg.annotation)}"
                        args.append(arg_str)
                    
                    # Handle *args
                    if item.args.vararg:
                        args.append(f"*{item.args.vararg.arg}")
                    
                    # Handle **kwargs
                    if item.args.kwarg:
                        args.append(f"**{item.args.kwarg.arg}")
                    
                    signature = f"{item.name}({', '.join(args)})"
                    class_info["methods"].append(signature)
            
            result["classes"].append(class_info)
        
        elif isinstance(node, ast.FunctionDef) or isinstance(node, ast.AsyncFunctionDef):
            # Extract function information
            # Build function signature
            args = []
            for arg in node.args.args:
                arg_str = arg.arg
                if arg.annotation:
                    arg_str += f": {ast.unparse(arg.annotation)}"
                args.append(arg_str)
            
            if node.args.vararg:
                args.append(f"*{node.args.vararg.arg}")
            if node.args.kwarg:
                args.append(f"**{node.args.kwarg.arg}")
            
            func_name = f"{node.name}({', '.join(args)})"
            
            # Get function lines (0-indexed in ast, convert to get actual lines)
            start_line = node.lineno - 1
            end_line = node.end_lineno
            func_lines = lines[start_line:end_line]
            
            # Build content with first 5 and last 5 lines
            if len(func_lines) <= 10:
                content = "\n".join(func_lines)
            else:
                first_five = func_lines[:5]
                last_five = func_lines[-5:]
                content = "\n".join(first_five) + "\n...\n" + "\n".join(last_five)
            
            result["functions"].append({
                "name": func_name,
                "content": content
            })
    
    return result


## 9. offline_pipeline.py

In [14]:

def offline_pipeline_file(prompt: str,
                          local_project_path: str, local_issues_path: str,
                          local_file_path: str, top_k: int = 2) -> AIMessage:
    log_and_print(f"Running offline_pipeline_file method...\n \
                      Args: local_project_path={local_project_path}\n \
                      | local_issues_path={local_issues_path} | local_file_path={local_file_path}\n \
                      | top_k={top_k}\n")

    try:
        os.stat(local_file_path)
        os.stat(local_issues_path)
        os.stat(local_project_path)
    except (OSError, ValueError) as e:
        log_and_print("Error occured in args of offline_pipeline_file")
        log_and_print(e)
        log_and_print(traceback.format_exc())
        raise e
    
    log_and_print(f"Reading file {local_file_path}...")
    file_content = get_file_content(local_file_path)

    context = context_retriever(local_project_path, local_issues_path, file_content, top_k)
    log_and_print(f"Numbering file content...")
    numbered_content = add_line_numbers(file_content)

    log_and_print(f"Creating user payload (aka prompt)...")
    user_payload = (
        prompt
        + "\n\n[CODE FILE PATH]\n"
        + local_file_path
        + "\n\n[BEGIN SUBMITTED CODE CONTENT]\n"
        + numbered_content
        + "\n[END SUBMITTED CODE CONTENT]\n\n"
        + context
    )
    log_and_print(f"{user_payload}")

    structured_llm = LLM.with_structured_output(PatchSuggestions)
    
    log_and_print(f"Calling LLM for patch generation with system prompt...\n{SYSTEM_PROMPT_PYTHON_PROGRAMMER}")
    response = structured_llm.invoke(
        [
            ("system", SYSTEM_PROMPT_PYTHON_PROGRAMMER),
            ("human", user_payload),
        ]
    )

    log_and_print(f"vvvvvvvvvvvv Response recived vvvvvvvvvvvv\n\n")
    log_and_print("\n--- Generated Snippets ---")
    for suggestion in response.edits:
        log_and_print(f"vvvvvvvvvvvvvvvvvvvvvvvv\n")
        log_and_print(f"{suggestion}")
    return response


def offline_pipeline_issue(prompt: str,
                           local_project_path: str, 
                           local_issues_path: str,
                           issue_content: str,
                           top_k: int = 2,
                           output_format: BaseModel = PatchSuggestions,
                           output_prompt: str = OUTPUT_FORMAT,
                           file_selection: bool = False) -> PatchSuggestions | DiffViewEdits:
    # issue_content = io_utils.get_issue_content(issue_id, local_issues_path)
    # issue_content = f"{issue_content[0]}\n{issue_content[1]}\n{issue_content[2]}" 

    if file_selection:
        file_docs = build_context_payload_from_docs(retriever_bm25_docs(local_project_path, local_issues_path, issue_content, top_k))
        structured_llm = LLM.with_structured_output(FilesToEdit)

        user_payload = (
            USER_PROMPT_FILESTOEDIT
            + "\n\n[ISSUE STATEMENT]\n"
            + issue_content
            + "\n[END ISSUE STATEMENT]\n\n"
            + "Top-30 similar files:\n"
            + file_docs
            + f"\n[PROJECT TREE]\n{get_project_tree(local_project_path)}\n[END PROJECT TREE]\n"
            + OUTPUT_FORMAT_FILESTOEDIT
        )
        log_and_print("Calling LLM for file selection...\n")    
        log_and_print(f"Prompt Sent:\n\n{user_payload}\n\n")
        retriever_response = structured_llm.invoke(
            [
                ("system", SYSTEM_PROMPT_PYTHON_PROGRAMMER),
                ("human", user_payload),
            ]
        )
        files_for_editing = "Only make changes to these files to solve the stated issue.\nFiles For Editing:\n"
        log_and_print(f"vvvvvvvvvvvv Retriever Response recived vvvvvvvvvvvv\n\n")
        log_and_print("\n--- Files to Edit ---")
        for file_path in retriever_response.files_for_editing:
            log_and_print(f">>>{file_path}")
            files_for_editing += f"- File Path:\n{file_path}\n[File Content START]\n{get_file_content(f"{local_project_path}/{file_path}")}\n[File Content END]\n"

    context = context_retriever(local_project_path, local_issues_path, issue_content, top_k)

    user_payload = (
        prompt 
        + "\n\n[ISSUE STATEMENT]\n"
        + issue_content
        + "\n[END ISSUE STATEMENT]\n\n"
        + files_for_editing if file_selection else ""
        + context
        + output_prompt
    )
    
    structured_llm = LLM.with_structured_output(output_format)

    log_and_print("Calling LLM...\n")
    response = structured_llm.invoke(
        [
            ("system", SYSTEM_PROMPT_PYTHON_PROGRAMMER),
            ("human", user_payload),
        ]
    )
    log_and_print(f"Prompt Sent:\n\n{user_payload}\n\n")
    log_and_print("Recieving response...\n")
    log_and_print("\n--- Generated Snippets ---")
    # for suggestion in response.edits:
    #     log_and_print(f"{suggestion}")
    log_and_print(f"{response.patch}")
    # patch_output.pretty_print_response(response)
    return response

## 10. bugsinpy_bugs.py

In [15]:


def _apply_patch_with_stats(patch: PatchSuggestions, buggy_file_content: str) -> tuple[str, dict[str, Any]]:
    """
    Internal helper that applies a PatchSuggestions object and also returns per-edit application stats.
    """
    newline = "\r\n" if "\r\n" in buggy_file_content else "\n"
    original_had_trailing_newline = buggy_file_content.endswith(("\n", "\r\n"))

    lines = buggy_file_content.splitlines(keepends=True)

    edits_sorted = sorted(
        patch.edits,
        key=lambda e: (max(1, int(getattr(e, "line_start_for_editing", 1))), max(1, int(getattr(e, "line_end_for_editing", 1)))),
        reverse=True,
    )

    stats: dict[str, Any] = {
        "num_edits_suggested": len(patch.edits),
        "num_edits_applied": 0,
        "num_edits_skipped": 0,
        "num_edits_failed": 0,
        "edit_results": [],
    }

    def _coerce_newlines(text: str) -> str:
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        if newline != "\n":
            text = text.replace("\n", newline)
        return text

    for edit in edits_sorted:
        try:
            start = int(getattr(edit, "line_start_for_editing", 1) or 1)
            end = int(getattr(edit, "line_end_for_editing", start) or start)
            if start < 1:
                start = 1
            if end < 1:
                end = 1
            if end < start:
                start, end = end, start

            # Convert to 0-based indices; end is inclusive in the edit model.
            start_idx = start - 1
            end_idx = min(len(lines), end)

            region_text = "".join(lines[start_idx:end_idx])
            expected = _coerce_newlines(getattr(edit, "exact_existing_buggy_snippet", "") or "")
            replacement = _coerce_newlines(getattr(edit, "correct_replacement_snippet", "") or "")

            applied = False
            method = None

            # Preferred: replace expected snippet within the specified region.
            if expected and expected in region_text:
                new_region = region_text.replace(expected, replacement, 1)
                applied = True
                method = "region_snippet_replace"
            # Next: if the entire region matches expected (after trimming only trailing newlines), replace region.
            elif expected and region_text.rstrip("\r\n") == expected.rstrip("\r\n"):
                new_region = replacement
                applied = True
                method = "region_whole_replace"
            # Fallback: replace the entire region by coordinates.
            else:
                new_region = replacement
                applied = True
                method = "region_coordinate_replace"

            # Preserve region's trailing newline presence to avoid accidental concatenation
            if region_text.endswith(newline) and new_region and not new_region.endswith(newline):
                new_region = new_region + newline
            if not region_text.endswith(newline) and new_region.endswith(newline) and end_idx == len(lines) and not original_had_trailing_newline:
                # If original file had no trailing newline, avoid adding one via last-region replacement.
                new_region = new_region[: -len(newline)]

            new_region_lines = new_region.splitlines(keepends=True)

            lines[start_idx:end_idx] = new_region_lines

            stats["num_edits_applied"] += 1
            stats["edit_results"].append(
                {
                    "category": getattr(edit, "category", None),
                    "confidence": getattr(edit, "confidence", None),
                    "line_start_for_editing": start,
                    "line_end_for_editing": end,
                    "applied": True,
                    "method": method,
                }
            )
        except Exception as ex:
            stats["num_edits_failed"] += 1
            stats["edit_results"].append(
                {
                    "category": getattr(edit, "category", None),
                    "confidence": getattr(edit, "confidence", None),
                    "line_start_for_editing": getattr(edit, "line_start_for_editing", None),
                    "line_end_for_editing": getattr(edit, "line_end_for_editing", None),
                    "applied": False,
                    "error": str(ex),
                }
            )

    out = "".join(lines)
    # Preserve original trailing newline if present
    if original_had_trailing_newline and not out.endswith(("\n", "\r\n")):
        out += newline

    return out, stats


def extract_bugsinpy():
    projects = {}
    with jsonlines.open("bugsinpy_bugs.jsonl") as reader:
        for bug in reader:
            # log_and_print(f"----------> {bug}")
            if bug["project"] not in projects.keys():
                projects[bug["project"]] = []
            projects[bug["project"]].append({"url": bug["url"], "id": bug["id"], "bug_commit": bug["bug_commit"], "fix_commit": bug["fix_commit"], "file_path": bug["file_path"]})
    return projects


def apply_patch(patch: PatchSuggestions, buggy_file_content: str) -> str:
    """
    Apply the generated edits to the file content to get the full generated file
    
    :param patch: LLM response object of the edits to be made
    :type patch: PatchSuggestions
    :param buggy_file_content: full file content before repair
    :type buggy_file_content: str
    :return: generated fixed file
    :rtype: str
    """
    generated, _ = _apply_patch_with_stats(patch, buggy_file_content)
    return generated


def compare(fixed_file_content: str, patch: PatchSuggestions, buggy_file_content: str, project_name: str, bug_id: int) -> dict[str, Any]:
    """
    Compare the actual real world fix with generated fix and recording their results
    
    :param fixed_file_content: real world file content after fix
    :type fixed_file_content: str
    :param patch: LLM response object of the edits to be made to the buggy file
    :type patch: PatchSuggestions
    :param buggy_file_content: full file content before repair
    :type buggy_file_content: str
    :return: key value pairs of metrics collected for this specific bug
    :rtype: dict[str, Any]
    """

    def _sha256(s: str) -> str:
        return hashlib.sha256(s.encode("utf-8", errors="replace")).hexdigest()

    def _line_diff_stats(a: str, b: str) -> dict[str, int]:
        a_lines = a.splitlines()
        b_lines = b.splitlines()
        sm = difflib.SequenceMatcher(a=a_lines, b=b_lines)
        additions = deletions = replaces = 0
        for tag, i1, i2, j1, j2 in sm.get_opcodes():
            if tag == "insert":
                additions += (j2 - j1)
            elif tag == "delete":
                deletions += (i2 - i1)
            elif tag == "replace":
                replaces += max(i2 - i1, j2 - j1)
        return {"additions": additions, "deletions": deletions, "replacements": replaces}

    def _unified_diff(a: str, b: str, fromfile: str, tofile: str, max_chars: int = 8000) -> str:
        diff = "\n".join(
            difflib.unified_diff(
                a.splitlines(),
                b.splitlines(),
                fromfile=fromfile,
                tofile=tofile,
                lineterm="",
            )
        )
        if len(diff) > max_chars:
            return diff[:max_chars] + "\n... (diff truncated)"
        return diff

    generated_fix, apply_stats = _apply_patch_with_stats(patch, buggy_file_content)

    similarity = difflib.SequenceMatcher(a=fixed_file_content, b=generated_fix).ratio()

    result: dict[str, Any] = {
        "project": project_name,
        "bug_id": bug_id,
        "file_path_to_edit": getattr(patch, "file_path_to_edit", None),
        "llm": LLM.get_config_jsonschema(),
        "num_edits_suggested": apply_stats.get("num_edits_suggested", len(getattr(patch, "edits", []) or [])),
        "num_edits_applied": apply_stats.get("num_edits_applied", 0),
        "num_edits_failed": apply_stats.get("num_edits_failed", 0),
        "generated_equals_actual": generated_fix == fixed_file_content,
        "buggy_equals_actual": buggy_file_content == fixed_file_content,
        "buggy_equals_generated": buggy_file_content == generated_fix,
        "similarity_generated_vs_actual": similarity,
        "sha256_buggy": _sha256(buggy_file_content),
        "sha256_generated": _sha256(generated_fix),
        "sha256_actual": _sha256(fixed_file_content),
        "diff_buggy_to_actual_stats": _line_diff_stats(buggy_file_content, fixed_file_content),
        "diff_buggy_to_generated_stats": _line_diff_stats(buggy_file_content, generated_fix),
        "diff_generated_to_actual_stats": _line_diff_stats(generated_fix, fixed_file_content),
        "diff_generated_to_actual": _unified_diff(
            generated_fix,
            fixed_file_content,
            fromfile=f"{project_name}-{bug_id}:generated",
            tofile=f"{project_name}-{bug_id}:actual",
        ),
        "edit_results": apply_stats.get("edit_results", []),
        "edits_summary": [
            {
                "category": getattr(e, "category", None),
                "confidence": getattr(e, "confidence", None),
                "line_start_for_editing": getattr(e, "line_start_for_editing", None),
                "line_end_for_editing": getattr(e, "line_end_for_editing", None),
            }
            for e in (getattr(patch, "edits", []) or [])
        ],
    }

    return result


def process_bugsinpy():
    projects_info = extract_bugsinpy()
    for project_name, bugs_list in projects_info.items():
        project_name = project_name.lower()
        
        project_path = "projects/" + project_name
        
        git_url = bugs_list[0].get("url")
        download_codebase(git_url)
        
        for bug in bugs_list:
            bug_commit = bug["bug_commit"]
            file_path = f"{project_path}/{bug["file_path"]}"
            download_codebase(git_url, bug_commit)
            results_path = os.path.join("results", f"{project_name}_results_{get_commit_date_posix(git_url, bug_commit)}_{LLM.get_name()}.jsonl")
            buggy_file_content = get_file_content(file_path)
            issues_filename, comments_filename = save_issues_and_comments_before_commit(git_url, bug_commit, issues_filename=project_name, comments_filename=project_name, include_prs=True)
            resp = offline_pipeline_file(OUTPUT_FORMAT + USER_PROMPT_V7, project_path, "issues/" + issues_filename, file_path, top_k=3)
            
            download_codebase(git_url, bug["fix_commit"])
            fixed_file = get_file_content(file_path)
            result = compare(fixed_file, resp, buggy_file_content, project_name, bug["id"])
            log_and_print(f"################ Results recorded for {project_name}-{bug["id"]} ################")
            log_and_print(result)
            with jsonlines.open(results_path, mode="w") as writer:
                writer.write(result)



## 11. swe_bench.py

In [16]:

# from summarize_discussions import summarize_all_discussions 


@dataclass(frozen=True)
class _ResolvedEdit:
    """Internal: edit with resolved 0-based [start, end) line slice."""
    start: int
    end: int
    snippet: "PatchSnippet"


def _split_lines_keepends(s: str) -> List[str]:
    return s.splitlines(keepends=True)


def _ensure_trailing_newline_like(original: str, replacement: str) -> str:
    if original.endswith("\n") and not replacement.endswith("\n"):
        return replacement + "\n"
    return replacement


def _find_unique_substring(haystack: str, needle: str) -> Tuple[int, int]:
    """Return (start_idx, end_idx) of a unique occurrence of needle in haystack."""
    if not needle:
        raise ValueError("exact_existing_buggy_snippet is empty; cannot locate edit.")
    
    first = haystack.find(needle)
    if first != -1:
        second = haystack.find(needle, first + 1)
        if second != -1:
            raise ValueError("exact_existing_buggy_snippet occurs multiple times; ambiguous.")
        return first, first + len(needle)
    
    def normalize(s: str) -> str:
        lines = s.splitlines(keepends=True)
        return "".join(line.lstrip() for line in lines)
    
    normalized_haystack = normalize(haystack)
    normalized_needle = normalize(needle)
    
    first = normalized_haystack.find(normalized_needle)
    if first != -1:
        hay_lines = haystack.splitlines(keepends=True)
        norm_hay_lines = [line.lstrip() for line in hay_lines]
        
        char_count = 0
        start_line = 0
        for i, line in enumerate(norm_hay_lines):
            if char_count + len(line) > first:
                start_line = i
                break
            char_count += len(line)
        
        needle_lines = needle.strip().splitlines()
        for i in range(len(hay_lines)):
            match = True
            for j, needle_line in enumerate(needle_lines):
                if i + j >= len(hay_lines):
                    match = False
                    break
                if hay_lines[i + j].strip() != needle_line.strip():
                    match = False
                    break
            if match:
                start_pos = sum(len(hay_lines[k]) for k in range(i))
                end_pos = sum(len(hay_lines[k]) for k in range(i + len(needle_lines)))
                return start_pos, end_pos
    
    from difflib import SequenceMatcher
    hay_lines = haystack.splitlines(keepends=True)
    needle_lines = needle.strip().splitlines()
    
    best_ratio = 0.0
    best_start = -1
    best_end = -1
    
    for i in range(len(hay_lines) - len(needle_lines) + 1):
        window = "".join(hay_lines[i:i + len(needle_lines)])
        ratio = SequenceMatcher(None, window.strip(), needle.strip()).ratio()
        if ratio > best_ratio:
            best_ratio = ratio
            best_start = i
            best_end = i + len(needle_lines)
    
    if best_ratio >= 0.7 and best_start >= 0:
        log_and_print(f"[FUZZY MATCH] Found snippet with {best_ratio:.1%} similarity at lines {best_start+1}-{best_end}")
        start_pos = sum(len(hay_lines[k]) for k in range(best_start))
        end_pos = sum(len(hay_lines[k]) for k in range(best_end))
        return start_pos, end_pos
    
    log_and_print(f"[DEBUG] Could not find snippet in file. Best match ratio: {best_ratio:.1%}")
    log_and_print(f"[DEBUG] Looking for:\n{needle[:200]}...")
    log_and_print(f"[DEBUG] File has {len(hay_lines)} lines. First 500 chars:\n{haystack[:500]}...")
    
    raise ValueError("exact_existing_buggy_snippet not found in file content.")


def _char_span_to_line_slice(text: str, start: int, end: int) -> Tuple[int, int]:
    """
    Convert a character span to a (start_line, end_line_exclusive) 0-based slice.
    Line boundaries are computed on '\n'.
    """
    start_line = text.count("\n", 0, start)
    end_line = text.count("\n", 0, end)
    
    if end > 0 and end <= len(text) and (end == len(text) or text[end - 1] != "\n"):
        end_line += 1
    return start_line, max(end_line, start_line)


def _resolve_edits(
    patch: "PatchSuggestions",
    original_content: str,
    *,
    min_confidence: float = -0.1,
    strict_line_range_match: bool = False,
) -> List[_ResolvedEdit]:
    """
    Resolve each PatchSnippet to a concrete 0-based line slice [start, end).
    If the provided line slice doesn't match, fall back to locating the snippet
    uniquely in the file (unless strict_line_range_match=True).
    """
    lines = _split_lines_keepends(original_content)
    resolved: List[_ResolvedEdit] = []

    for e in patch.edits:
        if e.confidence < min_confidence:
            continue

        start = max(0, e.line_start_for_editing - 1)
        end_excl = max(start, e.line_end_for_editing)

        in_bounds = start <= len(lines) and end_excl <= len(lines)
        slice_text = "".join(lines[start:end_excl]) if in_bounds else ""

        
        range_matches = (slice_text == e.exact_existing_buggy_snippet)
        if not range_matches and slice_text.strip() == e.exact_existing_buggy_snippet.strip():
            range_matches = not strict_line_range_match

        if range_matches and in_bounds:
            resolved.append(_ResolvedEdit(start=start, end=end_excl, snippet=e))
            continue

        if strict_line_range_match:
            raise ValueError(
                f"Line range {e.line_start_for_editing}-{e.line_end_for_editing} does not "
                f"match exact_existing_buggy_snippet for {patch.file_path_to_edit}."
            )

        c0, c1 = _find_unique_substring(original_content, e.exact_existing_buggy_snippet)
        s_line, e_line_excl = _char_span_to_line_slice(original_content, c0, c1)
        resolved.append(_ResolvedEdit(start=s_line, end=e_line_excl, snippet=e))

    resolved.sort(key=lambda r: (r.start, r.end), reverse=True)

    for i in range(len(resolved) - 1):
        a = resolved[i]
        b = resolved[i + 1]
        if b.end > a.start:
            raise ValueError(
                f"Overlapping edits detected: [{b.start},{b.end}) overlaps [{a.start},{a.end})."
            )

    return resolved


def apply_patch_suggestions_and_diff(
    patch: "PatchSuggestions",
    original_file_content: str,
    *,
    context_lines: int = 3,
    min_confidence: float = -0.1,
    strict_line_range_match: bool = False,
    fromfile: Optional[str] = None,
    tofile: Optional[str] = None,
) -> str:
    """
    Inputs:
      1) patch: PatchSuggestions (for a single file)
      2) original_file_content: content from the repo at the SWE-bench(-Lite) base commit

    Output:
      git-style unified diff between original and patched content.
    """
    path = patch.file_path_to_edit
    a_name = fromfile or f"a/{path}"
    b_name = tofile or f"b/{path}"

    resolved = _resolve_edits(
        patch,
        original_file_content,
        min_confidence=min_confidence,
        strict_line_range_match=strict_line_range_match,
    )

    lines = _split_lines_keepends(original_file_content)

    for r in resolved:
        old_block = "".join(lines[r.start:r.end])
        expected = r.snippet.exact_existing_buggy_snippet

        if expected and (old_block != expected):
            if expected in old_block:
                new_block = old_block.replace(
                    expected,
                    _ensure_trailing_newline_like(expected, r.snippet.correct_replacement_snippet),
                    1,
                )
                lines[r.start:r.end] = _split_lines_keepends(new_block)
                continue
            raise ValueError(
                f"Resolved edit block does not match exact_existing_buggy_snippet for {path}.\n"
                f"Edit category={r.snippet.category}, confidence={r.snippet.confidence}"
            )

        replacement = _ensure_trailing_newline_like(old_block, r.snippet.correct_replacement_snippet)
        lines[r.start:r.end] = _split_lines_keepends(replacement)

    patched_content = "".join(lines)

    diff_lines = list(
        unified_diff(
            _split_lines_keepends(original_file_content),
            _split_lines_keepends(patched_content),
            fromfile=a_name,
            tofile=b_name,
            n=context_lines,
            lineterm="",
            )
    )

    if not diff_lines:
        return f"diff --git {a_name} {b_name}\n"

    return "diff --git {a} {b}\n{rest}\n".format(
        a=a_name, b=b_name, rest="\n".join(diff_lines).rstrip("\n")
    )


def get_patch(response: PatchSuggestions, local_file_path: str):
    
    if not os.path.exists(local_file_path):
        log_and_print(f"[ERROR] File not found: {local_file_path}")
        log_and_print(f"[DEBUG] Model suggested path: {response.file_path_to_edit}")
        
        project_dir = os.path.dirname(local_file_path) or "."
        if os.path.exists(project_dir):
            similar_files = [f for f in os.listdir(project_dir) if f.endswith('.py')]
            log_and_print(f"[DEBUG] Python files in {project_dir}: {similar_files[:10]}")
        raise FileNotFoundError(f"Model suggested file path does not exist: {local_file_path}")

    local_file_content = get_file_content(local_file_path)
    if not local_file_content:
        raise ValueError(f"File is empty or could not be read: {local_file_path}")
    
    log_and_print(f"[DEBUG] Attempting to patch file: {local_file_path} ({len(local_file_content)} chars, {len(local_file_content.splitlines())} lines)")
    
    model_patch = apply_patch_suggestions_and_diff(response, local_file_content)

    local_file_content = local_file_content.splitlines(keepends=True)
    model_patch = model_patch.splitlines(keepends=True)
    model_suggested_path = response.file_path_to_edit

    log_and_print(f">>>>>>>> Local file content:\n{local_file_content}\n<<<<<<<<<<<")
    log_and_print(f">>>>>>>> model patch content:\n{local_file_content}\n<<<<<<<<<<")
    log_and_print(f"Model suggested path: {response.file_path_to_edit}")
        

    diff = difflib.unified_diff(
        local_file_content, model_patch,
        fromfile=f"a/{model_suggested_path}",
        tofile=f"b/{model_suggested_path}"
    )
    diff = "".join(diff)
    diff = f"diff --git a/{model_suggested_path} b/{model_suggested_path}\n{diff}"

    log_and_print(f"#####@@@@@@##### DIFF VIEW #####@@@@@@#####\n{diff}")

    return diff


def run_swebench():
    local_file_path = "codebase/manage.py"
    # local_file_response = offline_pipeline_file(OUTPUT_FORMAT + USER_PROMPT_ISSUES, "codebase", "issues/local_issues_summaries.jsonl", local_file_path, top_k=5)
    # diff = apply_patch_suggestions_and_diff(local_file_response, get_file_content(local_file_path))

    swebench_lite_dev = load_dataset('princeton-nlp/SWE-bench_Lite', split='dev')


    results = []
    for task in swebench_lite_dev:
        # clone fresh repo at base_commit
        # repo = task["repo"]
        # commit = task["base_commit"]
        instance_id = task["instance_id"]
        model_name = LLM.model_name if hasattr(LLM, "model_name") else getattr(LLM, "model", None)

        f = False
        results_path = os.path.join("results", f"swe_bench_lite_results.jsonl")
        with jsonlines.open(results_path, mode="r") as reader:
            for bench in reader:
                if bench["instance_id"] == instance_id and bench["model_name_or_path"] == model_name:
                    f = True

        if f:
            log_and_print(f"XXXXXXXXXXXXXXXXXXXXX Skipping {instance_id} By {model_name} XXXXXXXXXXXXXXXXXXXXX")
            continue

        project_name = task["repo"]
        
        project_path = f"projects/" + project_name.split("/")[1]
        
        git_url = f"https://github.com/{task["repo"]}"
        download_codebase(git_url)
        
      
        bug_commit = task["base_commit"]
        download_codebase(git_url, bug_commit)
        issues_filename, comments_filename = save_issues_and_comments_before_commit(git_url, bug_commit, issues_filename=project_name, comments_filename=project_name, include_prs=True, max_workers=4)
        
        # file_path = f"{project_path}/{task["file_path"]}"
        # resp = offline_pipeline_file(OUTPUT_FORMAT + USER_PROMPT_V7, project_path, "issues/" + issues_filename, file_path, top_k=3)
        # summarize_all_discussions(
        #     issues_path="issues/" + issues_filename,
        #     comments_path="comments/" + comments_filename,
        #     output_path="summarized_issues/" + issues_filename,
        #     max_workers=2
        # )
        resp = offline_pipeline_issue(USER_PROMPT_MULTIFILE_ZERO_SHOT, \
                                            project_path, "issues/" + issues_filename, task["problem_statement"], top_k=3, \
                                            output_format=DiffViewEdits, output_prompt=OUTPUT_FORMAT_Diffview, \
                                            file_selection=True)

        # resp = offline_pipeline_issue(USER_PROMPT_MULTIFILE_ZERO_SHOT, project_path, "issues/" + issues_filename, task["problem_statement"], top_k=3, output_format=DiffViewEdits, output_prompt=OUTPUT_FORMAT_Diffview)
        
        # patch = get_patch(resp, project_path + '/' + resp.file_path_to_edit)

        if validate_patch(resp.patch):
            results.append({"instance_id": task["instance_id"], "model_name_or_path": model_name, "model_patch": resp.patch})
        else:
            results.append({"instance_id": task["instance_id"], "model_name_or_path": model_name, "model_patch": ""})

        # diff = apply_patch_and_get_diff(resp, project_path + '/' + resp.file_path_to_edit)
        
        # log_and_print(LLM.model_dump())
        # log_and_print(LLM.get_output_jsonschema())

        with jsonlines.open(f"{DRIVE_PATH}/" + results_path, mode="a") as writer:
            writer.write(results[-1])

        with jsonlines.open(results_path, mode="a") as writer:
            writer.write(results[-1])
            log_and_print(f"################ Results recorded for {project_name}-{task["instance_id"]} ################")
            # writer.write_all(results)


## 12. Run SWE-bench

After loading all modules above, run the main function:

In [17]:
# Run SWE-bench pipeline
# This will only work after all modules above are properly loaded

try:
    run_swebench()
except NameError:
    print("⚠️  Please load all required modules first (cells 5-11)")
    print("The swe_bench module and its dependencies must be loaded before running.")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 37d2b870-e13b-4eb8-a086-1d003f7d4a20)')' thrown while requesting HEAD https://huggingface.co/datasets/princeton-nlp/SWE-bench_Lite/resolve/main/README.md
Retrying in 1s [Retry 1/5].


✓ Successfully checked out to commit main in existing repo
✓ Successfully checked out to commit 14e1a23 in existing repo
====== Fetching commit 14e1a23 ======
Commit date: 2021-10-13 14:14:42+00:00
Earliest date (6 months back): 2021-04-16 14:14:42+00:00
Filtering issues and comments created between 2021-04-16 14:14:42+00:00 and 2021-10-13 14:14:42+00:00...
Found 653 already processed issues, will skip them.
Found 1630 already processed comments, will skip them.

====== Collecting issues before commit... ======
Searching for issues within date range using GitHub Search API...
Search query: repo:sqlfluff/sqlfluff created:2021-04-16..2021-10-13
Found 21 issues to process (after filtering already processed)


✓ Added 0 new issues (skipped 0 existing)
✓ Added 0 new comments (skipped 0 existing)
Issues saved to: issues/sqlfluff/sqlfluff1634134482.0.jsonl
Comments saved to: comments/sqlfluff/sqlfluff1634134482.0.jsonl
Checking project root dir: projects/sqlfluff
Checking file: projects/sqlfl

NotFoundError: <html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx</center>
<script>(function(){function c(){var b=a.contentDocument||a.contentWindow.document;if(b){var d=b.createElement('script');d.innerHTML="window.__CF$cv$params={r:'9cb2d3b77c3a92b9',t:'MTc3MDYzNDQ0OS4wMDAwMDA='};var a=document.createElement('script');a.nonce='';a.src='/cdn-cgi/challenge-platform/scripts/jsd/main.js';document.getElementsByTagName('head')[0].appendChild(a);";b.getElementsByTagName('head')[0].appendChild(d)}}if(document.body){var a=document.createElement('iframe');a.height=1;a.width=1;a.style.position='absolute';a.style.top=0;a.style.left=0;a.style.border='none';a.style.visibility='hidden';document.body.appendChild(a);if('loading'!==document.readyState)c();else if(window.addEventListener)document.addEventListener('DOMContentLoaded',c);else{var e=document.onreadystatechange||function(){};document.onreadystatechange=function(b){e(b);'loa